In [1]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
parent_dir = os.path.dirname(os.environ["GTE_DIR"].replace("Glaciation_time_estimator",""))
GTE_DIR=os.environ["GTE_DIR"]
sys.path.insert(0, parent_dir)
from Glaciation_time_estimator.Data_postprocessing.Job_result_fp_generator import generate_tracking_filenames
from Glaciation_time_estimator.Auxiliary_func.config_reader import read_config

In [2]:
config = read_config(
    os.path.join(GTE_DIR,'config_half.yaml'))
analyze_year=True
year=2022
glac_threshold=0.4
global global_rmse
global_rmse = config["Global_sqrt_mse"]
classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
# temperature_palette = ['#eff3ff','#c6dbef','#9ecae1','#6baed6','#3182bd','#08519c']
temperature_palette = ['#f1eef6','#d0d1e6','#a6bddb','#74a9cf','#2b8cbe','#045a8d']
classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
temperature_palette = ['#eff3ff','#c6dbef','#9ecae1','#6baed6','#3182bd','#08519c']

In [4]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'DJF'
    elif month in [3, 4, 5]:
        return 'MAM'
    elif month in [6, 7, 8]:
        return 'JJA'
    else:
        return 'SON'
if os.uname()[1]=="n2o":
    combined_cloud_df = pd.read_parquet(f"/wolke_scratch/dnikolo/Final_results/{year}_all.parquet")
    glaciations_df_04 = pd.read_parquet(f"/wolke_scratch/dnikolo/Final_results/{year}_test.parquet")
    glaciations_df_03 = pd.read_parquet(f"/wolke_scratch/dnikolo/Final_results/{year}_glac_03.parquet")
if os.uname()[1][:3]=="eu-":
    combined_cloud_df = pd.read_parquet(f"/cluster/work/climate/dnikolo/Cloud_analysis/full_years/{year}_all.parquet")
    glaciations_df_04 = pd.read_parquet(f"/cluster/work/climate/dnikolo/Cloud_analysis/full_years/{year}_glac_03_thresh.parquet") #_{int(glac_threshold*10):02}_thresh

combined_cloud_df=combined_cloud_df[~combined_cloud_df.is_large_pix_cloud]
combined_cloud_df = combined_cloud_df[(combined_cloud_df.avg_lat >30) | (combined_cloud_df.avg_lat<-30)]
combined_cloud_df['Season'] = combined_cloud_df['track_start_time'].dt.month.apply(month_to_season)


glaciations_df_04=glaciations_df_04[~glaciations_df_04.is_large_pix_cloud]
glaciations_df_04 = glaciations_df_04[(glaciations_df_04.avg_lat >30) | (glaciations_df_04.avg_lat<-30)]
glaciations_df_04["Radius [km]"]=np.sqrt(glaciations_df_04["avg_size[km]"]/np.pi)
glaciations_df_04['Season'] = glaciations_df_04['track_start_time'].dt.month.apply(month_to_season)
glaciating_clouds_04 = glaciations_df_04.drop_duplicates(subset="Cloud_ID",keep="first")
combined_cloud_df['is_glaciating_04'] = combined_cloud_df.index.isin(glaciating_clouds_04['Cloud_ID'])
assert glaciating_clouds_04["Cloud_ID"].isin(combined_cloud_df.index).all(),"The files don't correspont to each other"

glaciations_df_03=glaciations_df_03[~glaciations_df_03.is_large_pix_cloud]
glaciations_df_03 = glaciations_df_03[(glaciations_df_03.avg_lat >30) | (glaciations_df_03.avg_lat<-30)]
glaciations_df_03["Radius [km]"]=np.sqrt(glaciations_df_03["avg_size[km]"]/np.pi)
glaciations_df_03['Season'] = glaciations_df_03['track_start_time'].dt.month.apply(month_to_season)
glaciating_clouds_03 = glaciations_df_03.drop_duplicates(subset="Cloud_ID",keep="first")
combined_cloud_df['is_glaciating_03'] = combined_cloud_df.index.isin(glaciating_clouds_03['Cloud_ID'])
assert glaciating_clouds_03["Cloud_ID"].isin(combined_cloud_df.index).all(),"The files don't correspont to each other"

In [15]:
glaciations_df_03["has_04_glac"] = glaciations_df_03["Cloud_ID"].isin(glaciating_clouds_04['Cloud_ID'])

In [17]:
glaciations_df_03["has_04_glac"].sum()/len(glaciations_df_04)

1.012978501563974